# Qualidade de `campanha_ago26.csv`

Diagnóstico do único experimento controlado da base pelos mesmos cinco critérios usados nos outros arquivos:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Quem deveria estar no sorteio está, e o que foi prometido (cupom) aparece? |
| **Validade** | Grupos, canais, títulos, ofertas e engajamento estão no domínio esperado? |
| **Consistência** | O sorteio está equilibrado? O controle ficou sem nada? Oferta, plano e engajamento concordam? |
| **Unicidade** | Cada assinante aparece uma vez só? |
| **Temporalidade** | A janela entre o envio e o corte permite medir o efeito? O engajamento tem data? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).

**Decisões já registradas que valem aqui** (`decisoes.md`): contas de teste e cópias de `usuarios.csv` removidas;
datas dos logs em horário de Brasília; nenhuma informação posterior ao corte. As situações das assinaturas vêm das
tabelas já limpas em `dados/processados/` (notebooks 05, 06 e 09) e as cobranças de `pagamentos_limpo.parquet`.

In [1]:
import hashlib
import math
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
DIA = pd.Timestamp("2026-08-10")
ENVIO = pd.Timestamp("2026-08-10 10:00:00")   # horário de Brasília (o dicionário não diz o fuso)
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)
GRUPOS = ["controle", "conteudo", "desafio", "cupom"]

ARQUIVOS = ["campanha_ago26.csv", "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    # status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

def qui2(tabela):
    # Qui-quadrado de independência (ou de aderência, para uma linha só), calculado à mão (scipy não está instalado).
    # p-valor pela aproximação de Wilson-Hilferty, boa para os graus de liberdade usados aqui.
    t = pd.DataFrame(tabela).astype(float)
    if t.shape[0] == 1:
        esperado = pd.DataFrame([[t.values.sum() / t.shape[1]] * t.shape[1]], columns=t.columns)
        gl = t.shape[1] - 1
    else:
        esperado = pd.DataFrame(t.sum(axis=1).values[:, None] * t.sum(axis=0).values[None, :] / t.values.sum(),
                                index=t.index, columns=t.columns)
        gl = (t.shape[0] - 1) * (t.shape[1] - 1)
    x = float(((t.values - esperado.values) ** 2 / esperado.values).sum())
    z = ((x / gl) ** (1 / 3) - (1 - 2 / (9 * gl))) / math.sqrt(2 / (9 * gl))
    return x, gl, 0.5 * math.erfc(z / math.sqrt(2))

## 0. Leitura

Tudo como texto e sem converter vazio em nulo. Das tabelas limpas, só as colunas que as checagens pedem.

In [2]:
campanha = pd.read_csv(BRUTOS / "campanha_ago26.csv", dtype=str, keep_default_na=False)
assert len(campanha) == 14_120  # cabeçalho da seção 05 do dicionário
N = len(campanha)
print(f"campanha: {N:,} linhas x {campanha.shape[1]} colunas -> {list(campanha.columns)}")
campanha.sample(5, random_state=0)

campanha: 14,120 linhas x 9 colunas -> ['user_id', 'grupo', 'enviado_em', 'canal', 'titulo_mensagem', 'oferta', 'abriu_push', 'abriu_email', 'clicou']


,user_id,grupo,enviado_em,canal,titulo_mensagem,oferta,abriu_push,abriu_email,clicou
3710,u_a176f87f,desafio,2026-08-10 10:00:00,push+email,Novo desafio de 6 semanas montado para você,,False,False,False
2465,u_4dc74d0b,controle,,,,,False,False,False
10644,u_a2ceb849,desafio,2026-08-10 10:00:00,push+email,Novo desafio de 6 semanas montado para você,,True,False,False
6016,u_d818b004,cupom,2026-08-10 10:00:00,push+email,Oferta para voltar a treinar,1_mes_gratis,False,False,False
33,u_4a11c2f8,conteudo,2026-08-10 10:00:00,push+email,Seu plano de treino personalizado está pronto,,False,False,False


### Colunas derivadas para a análise

- `enviado`: `enviado_em` convertida; `abriu_push_b`, `abriu_email_b`, `clicou_b`: engajamento como booleano;
- da assinatura (tabelas limpas): plano, canal de cobrança, início, fim, coorte anterior ao log e dias de assinatura
  no envio; renovação automática desligada antes do envio, pelo log;
- da conta (`usuarios_limpa.parquet`): relógio conectado e coorte de cadastro.

O arquivo original não muda.

In [3]:
c = campanha.copy()
c["enviado"] = pd.to_datetime(c["enviado_em"].replace("", None), format="%Y-%m-%d %H:%M:%S")
for col in ["abriu_push", "abriu_email", "clicou"]:
    c[f"{col}_b"] = c[col].map({"True": True, "False": False})

assinaturas = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet")
eventos = pd.read_parquet(PROCESSADOS / "eventos_assinatura_limpo.parquet",
                          columns=["user_id", "assinatura_id", "data_br", "evento", "plano"])
usuarios_limpa = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet",
                                 columns=["user_id", "relogio_conectado", "cadastro_antes_do_log"])

c = c.merge(assinaturas[["user_id", "assinatura_id", "plano_atual", "canal_cobranca", "inicio", "data_fim",
                         "inicio_antes_do_log"]], on="user_id", how="left", validate="one_to_one")
c = c.merge(usuarios_limpa, on="user_id", how="left", validate="one_to_one")
c["dias_de_assinatura"] = (DIA - c["inicio"]).dt.days

liga = eventos[eventos["evento"].isin(["renovacao_automatica_desligada", "renovacao_automatica_religada"])
               & (eventos["data_br"] < ENVIO)].sort_values(["user_id", "data_br"])
ultimo = liga.drop_duplicates("user_id", keep="last").set_index("user_id")["evento"]
c["renovacao_desligada_no_envio"] = c["user_id"].map(ultimo).eq("renovacao_automatica_desligada")
print("contas sem assinatura na tabela limpa:", c["plano_atual"].isna().sum(), "| sem cadastro limpo:", c["relogio_conectado"].isna().sum())

contas sem assinatura na tabela limpa: 0 | sem cadastro limpo: 0


## 1. Completude

O vazio do controle é estrutural (sorteado para não receber nada); nos grupos tratados, envio, canal e título devem
estar sempre preenchidos, e a oferta só no grupo cupom.

In [4]:
colunas = ["enviado_em", "canal", "titulo_mensagem", "oferta", "abriu_push", "abriu_email", "clicou"]
vazio_grupo = pd.DataFrame({col: (campanha[col].str.strip() == "").groupby(campanha["grupo"]).sum() for col in colunas}).loc[GRUPOS]
display(vazio_grupo)
VAZIOS_DISFARCADOS = {"nan", "null", "none", "na", "n/a", "-"}
disfarcado = int(sum(campanha[col].str.strip().str.lower().isin(VAZIOS_DISFARCADOS).sum() for col in campanha.columns))

tratado = c["grupo"] != "controle"
falta_tratado = int(vazio_grupo.loc[["conteudo", "desafio", "cupom"], ["enviado_em", "canal", "titulo_mensagem"]].values.sum())
falta_oferta = int(vazio_grupo.loc["cupom", "oferta"])
achado("Completude", "vazio disfarçado (nan, null, -)", f"{disfarcado}", disfarcado, "Nenhum")
achado("Completude", "envio, canal ou título vazio nos grupos tratados; oferta vazia no cupom",
       f"{falta_tratado + falta_oferta}", falta_tratado + falta_oferta, "Nenhum")
achado("Completude", "controle sem envio, canal, título nem oferta",
       f"as {int((c['grupo'] == 'controle').sum()):,} linhas do controle, nas quatro colunas: vazio estrutural", 0,
       "Nenhum: é o desenho do experimento", status="esperado")

,enviado_em,canal,titulo_mensagem,oferta,abriu_push,abriu_email,clicou
grupo,,,,,,,
controle,3579,3579,3579,3579,0,0,0
conteudo,0,0,0,3554,0,0,0
desafio,0,0,0,3515,0,0,0
cupom,0,0,0,0,0,0,0


In [5]:
brutos_usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False, usecols=["user_id", "email"])
email = brutos_usuarios.set_index("user_id")["email"].str.strip()
teste = set(email.index[email.str.lower().str.endswith("@vitaliza.com.br")])
copia = set(email.index[email != email.str.lower()])
n_teste, n_copia = int(c["user_id"].isin(teste).sum()), int(c["user_id"].isin(copia).sum())
sem_cad = int((~c["user_id"].isin(email.index)).sum())
achado("Completude", "contas de teste, cópias ou sem cadastro no sorteio", f"teste: {n_teste}; cópias: {n_copia}; sem cadastro: {sem_cad}",
       n_teste + n_copia + sem_cad, "Se houver: excluir")

### A população do sorteio

Número-prova: os 14.120 "assinantes pagantes daquele dia" (dicionário e enunciado). Reconstruo pelos dados já limpos
quem pagava em 10/08/2026, com quatro definições do "dia", e comparo nos dois sentidos.

In [6]:
ids_campanha = set(c["user_id"])
definicoes = {
    "início até 10/08 e fim em 10/08 ou depois": (assinaturas["inicio"] <= DIA) & (assinaturas["data_fim"].isna() | (assinaturas["data_fim"] >= DIA)),
    "início até 10/08 e fim depois de 10/08": (assinaturas["inicio"] <= DIA) & (assinaturas["data_fim"].isna() | (assinaturas["data_fim"] > DIA)),
    "início antes de 10/08 e fim em 10/08 ou depois": (assinaturas["inicio"] < DIA) & (assinaturas["data_fim"].isna() | (assinaturas["data_fim"] >= DIA)),
    "início antes de 10/08 e fim depois de 10/08": (assinaturas["inicio"] < DIA) & (assinaturas["data_fim"].isna() | (assinaturas["data_fim"] > DIA)),
}
pop = pd.DataFrame({nome: {"pagantes": int(m.sum()),
                           "do sorteio fora da definição": len(ids_campanha - set(assinaturas.loc[m, "user_id"])),
                           "pagantes fora do sorteio": len(set(assinaturas.loc[m, "user_id"]) - ids_campanha)}
                    for nome, m in definicoes.items()}).T
display(pop)

dia_todo = definicoes["início até 10/08 e fim em 10/08 ou depois"]
fora = assinaturas.loc[dia_todo & ~assinaturas["user_id"].isin(ids_campanha)]
exp_fora = eventos[eventos["user_id"].isin(fora["user_id"]) & (eventos["evento"] == "assinatura_expirada")]
print("pagantes do dia fora do sorteio:", len(fora), "| fim:", fora["data_fim"].dt.date.astype(str).tolist(),
      "| hora da expiração:", exp_fora["data_br"].dt.strftime("%H:%M").tolist())
achado("Completude", "população do sorteio contra os pagantes de 10/08 (número-prova 14.120)",
       f"as 14.120 contas estão entre os {int(dia_todo.sum()):,} que pagavam em algum momento de 10/08; "
       f"{len(fora)} pagantes do dia ficaram fora, ambos expirados em 10/08 (como outros 33 que entraram no sorteio)",
       0, "Nenhum: 2 contas em 14.122 não mudam o resultado; registrar como limite do arquivo", status="esperado")

,pagantes,do sorteio fora da definição,pagantes fora do sorteio
início até 10/08 e fim em 10/08 ou depois,14122,0,2
início até 10/08 e fim depois de 10/08,14087,33,0
início antes de 10/08 e fim em 10/08 ou depois,14074,48,2
início antes de 10/08 e fim depois de 10/08,14039,81,0


pagantes do dia fora do sorteio: 2 | fim: ['2026-08-10', '2026-08-10'] | hora da expiração: ['09:48', '13:37']


### O cupom aparece nas cobranças?

O enunciado promete 30% de desconto por 3 meses no mensal (R$ 39,90 vira R$ 27,93) e 1 mês grátis no anual. Se o cupom
foi usado, o mensal do grupo cupom deveria ter cobranças de R$ 27,93 depois do envio, e a renovação anual deveria vir
um mês depois do aniversário. Comparo com o controle, do mesmo plano.

In [7]:
pagamentos = pd.read_parquet(PROCESSADOS / "pagamentos_limpo.parquet", columns=["user_id", "data_br", "status", "valor"])
apos = pagamentos[(pagamentos["data_br"] >= ENVIO) & (pagamentos["data_br"] < FIM_CORTE) & (pagamentos["status"] == "aprovado")]
apos = apos.merge(c[["user_id", "grupo", "plano_atual"]], on="user_id")
valores_mensal = apos[apos["plano_atual"] == "mensal"].pivot_table(index="valor", columns="grupo", values="user_id",
                                                                   aggfunc="size", fill_value=0)[GRUPOS]
display(valores_mensal)
desconto = int((apos["valor"].round(2) == 27.93).sum())
print(f"cobranças de R$ 27,93 depois do envio: {desconto}")

grupo,controle,conteudo,desafio,cupom
valor,,,,
39.9,2196,2222,2274,2263


cobranças de R$ 27,93 depois do envio: 0


In [8]:
anual = c[c["plano_atual"] == "anual"].copy()
# primeiro aniversário da assinatura a partir de 10/08/2026 (nunca o próprio início)
anual["aniversario"] = [i + pd.DateOffset(years=max(1, DIA.year - i.year)) for i in anual["inicio"]]
antes = anual["aniversario"] < DIA
anual.loc[antes, "aniversario"] = [a + pd.DateOffset(years=1) for a in anual.loc[antes, "aniversario"]]
na_janela = anual[(anual["aniversario"] >= DIA) & (anual["aniversario"] <= CORTE)
                  & (anual["data_fim"].isna() | (anual["data_fim"] >= anual["aniversario"]))].copy()
ren = eventos[(eventos["evento"] == "renovacao") & (eventos["data_br"] >= ENVIO) & (eventos["data_br"] < FIM_CORTE)]
primeira_ren = ren.sort_values("data_br").drop_duplicates("user_id").set_index("user_id")["data_br"].dt.normalize()
na_janela["dias_ate_renovar"] = (na_janela["user_id"].map(primeira_ren) - na_janela["aniversario"]).dt.days
resumo_anual = na_janela.groupby("grupo").agg(
    aniversario_na_janela=("user_id", "size"),
    renovou_ate_3_dias=("dias_ate_renovar", lambda s: int(s.abs().le(3).sum())),
    renovou_25_a_35_dias_depois=("dias_ate_renovar", lambda s: int(s.between(25, 35).sum())),
    sem_renovacao_ate_o_corte=("dias_ate_renovar", lambda s: int(s.isna().sum())),
).loc[GRUPOS]
display(resumo_anual)
mes_gratis = int(resumo_anual.loc["cupom", "renovou_25_a_35_dias_depois"])
cupom_mensal = int(((c["grupo"] == "cupom") & (c["plano_atual"] == "mensal")).sum())
achado("Completude", "cupom prometido pelo enunciado não aparece nas cobranças",
       f"0 cobranças de R$ 27,93 depois do envio ({cupom_mensal:,} mensais receberam o cupom; os valores do grupo cupom são os "
       f"mesmos do controle); no anual, {mes_gratis} renovações do grupo cupom vieram ~1 mês depois do aniversário. "
       "Hipótese a confirmar com o dono do dado: o cupom não foi resgatado por ninguém, ou o resgate não chegou às cobranças",
       int((c["grupo"] == "cupom").sum()),
       "Medir o efeito do cupom como efeito da mensagem (intenção de tratar), sem custo de desconto observado; "
       "perguntar ao dono do dado se houve resgate antes de usar o custo do cupom na conta de retorno")

,aniversario_na_janela,renovou_ate_3_dias,renovou_25_a_35_dias_depois,sem_renovacao_ate_o_corte
grupo,,,,
controle,305,176,0,126
conteudo,273,156,0,115
desafio,303,166,0,133
cupom,281,141,0,137


## 2. Validade

Cada categoria contra o dicionário e o enunciado; envio convertendo; engajamento só `True`/`False`.

In [9]:
TITULOS = {"desafio": "Novo desafio de 6 semanas montado para você", "cupom": "Oferta para voltar a treinar",
           "conteudo": "Seu plano de treino personalizado está pronto", "controle": ""}
dominio = {
    "grupo": set(GRUPOS), "canal": {"push+email", ""}, "oferta": {"30pct_3_meses", "1_mes_gratis", ""},
    "abriu_push": {"True", "False"}, "abriu_email": {"True", "False"}, "clicou": {"True", "False"},
}
fora_dominio = {col: sorted(set(campanha[col]) - permitidos) for col, permitidos in dominio.items()}
print("fora do domínio:", fora_dominio)
n_fora = int(sum(campanha[col].isin(v).sum() for col, v in fora_dominio.items()))
achado("Validade", "grupo, canal, oferta e engajamento fora do domínio", f"{n_fora}", n_fora, "Nenhum")

titulo_errado = int((campanha["titulo_mensagem"] != campanha["grupo"].map(TITULOS)).sum())
achado("Validade", "título diferente do enunciado para o grupo", f"{titulo_errado}", titulo_errado, "Nenhum")

padroes = campanha["enviado_em"].str.replace(r"\d", "9", regex=True).value_counts().to_dict()
falha = int((c["enviado"].isna() & (campanha["enviado_em"] != "")).sum())
achado("Validade", "enviado_em que não converte", f"{falha}; padrões: {padroes}", falha, "Nenhum")
padrao_id = int((~campanha["user_id"].str.fullmatch(r"u_[0-9a-f]{8}")).sum())
achado("Validade", "user_id fora do padrão", f"{padrao_id}", padrao_id, "Nenhum")

fora do domínio: {'grupo': [], 'canal': [], 'oferta': [], 'abriu_push': [], 'abriu_email': [], 'clicou': []}


## 3. Consistência

### Equilíbrio do sorteio

Grupos "de tamanho parecido" e, por serem sorteados, com o mesmo perfil. Teste qui-quadrado: p-valor baixo (< 0,05)
indicaria sorteio desequilibrado.

In [10]:
tam = c["grupo"].value_counts().loc[GRUPOS]
x, gl, p = qui2(tam.to_frame().T)
print(f"tamanhos: {tam.to_dict()} | qui² = {x:.2f}, gl = {gl}, p = {p:.3f}")
achado("Consistência", "tamanho dos grupos (qui² de aderência)",
       f"{tam.to_dict()}; qui² = {x:.2f} com {gl} gl, p = {p:.2f}", 0,
       "Nenhum" if p >= 0.05 else "Investigar o sorteio", status="ok" if p >= 0.05 else "atenção")

tamanhos: {'controle': 3579, 'conteudo': 3554, 'desafio': 3515, 'cupom': 3472} | qui² = 1.86, gl = 3, p = 0.606


In [11]:
c["faixa_assinatura"] = pd.cut(c["dias_de_assinatura"], [-1, 90, 365, 730, 100_000],
                               labels=["até 3 meses", "3 a 12 meses", "1 a 2 anos", "mais de 2 anos"])
perfis = ["plano_atual", "canal_cobranca", "faixa_assinatura", "inicio_antes_do_log", "relogio_conectado",
          "renovacao_desligada_no_envio"]
linhas_perfil = []
for col in perfis:
    t = pd.crosstab(c[col], c["grupo"])[GRUPOS]
    x, gl, p = qui2(t)
    pct = (100 * t / t.sum()).round(1)
    linhas_perfil.append({"perfil": col, "qui²": round(x, 2), "gl": gl, "p": round(p, 3),
                          "maior diferença entre grupos (p.p.)": float((pct.max(axis=1) - pct.min(axis=1)).max())})
    display(pct.assign(perfil=col).set_index("perfil", append=True))
equilibrio = pd.DataFrame(linhas_perfil)
display(equilibrio)
mediana = c.groupby("grupo")["dias_de_assinatura"].median().loc[GRUPOS]
print("mediana de dias de assinatura no envio:", mediana.to_dict())
desequilibrados = equilibrio[equilibrio["p"] < 0.05]
achado("Consistência", "perfil dos grupos (plano, canal, tempo de assinatura, coorte, relógio, renovação desligada)",
       f"{len(equilibrio)} perfis testados; menor p = {equilibrio['p'].min():.2f}; maior diferença entre grupos "
       f"{equilibrio['maior diferença entre grupos (p.p.)'].max():.1f} p.p.", 0 if desequilibrados.empty else len(desequilibrados),
       "Nenhum" if desequilibrados.empty else "Ajustar a comparação pelo perfil desequilibrado",
       status="ok" if desequilibrados.empty else "atenção")

,grupo,controle,conteudo,desafio,cupom
plano_atual,perfil,,,,
anual,plano_atual,59.8,58.8,58.6,59.1
mensal,plano_atual,40.2,41.2,41.4,40.9


,grupo,controle,conteudo,desafio,cupom
canal_cobranca,perfil,,,,
app_store,canal_cobranca,36.8,37.1,36.6,35.7
google_play,canal_cobranca,33.5,32.0,33.8,34.9
web_cartao,canal_cobranca,20.4,20.8,20.5,19.8
web_pix,canal_cobranca,9.3,10.2,9.1,9.7


,grupo,controle,conteudo,desafio,cupom
faixa_assinatura,perfil,,,,
até 3 meses,faixa_assinatura,17.4,18.0,17.4,17.7
3 a 12 meses,faixa_assinatura,44.7,45.1,47.0,43.4
1 a 2 anos,faixa_assinatura,25.6,24.6,24.5,26.2
mais de 2 anos,faixa_assinatura,12.3,12.3,11.2,12.7


,grupo,controle,conteudo,desafio,cupom
inicio_antes_do_log,perfil,,,,
False,inicio_antes_do_log,89.2,88.6,90.1,88.6
True,inicio_antes_do_log,10.8,11.4,9.9,11.4


,grupo,controle,conteudo,desafio,cupom
relogio_conectado,perfil,,,,
apple_watch,relogio_conectado,18.5,19.7,19.1,17.8
garmin,relogio_conectado,19.0,19.5,19.4,20.0
nenhum,relogio_conectado,62.4,60.9,61.5,62.2


,grupo,controle,conteudo,desafio,cupom
renovacao_desligada_no_envio,perfil,,,,
False,renovacao_desligada_no_envio,83.3,83.1,83.1,83.5
True,renovacao_desligada_no_envio,16.7,16.9,16.9,16.5


,perfil,qui²,gl,p,maior diferença entre grupos (p.p.)
0,plano_atual,1.15,3,0.768,1.2
1,canal_cobranca,8.91,9,0.446,2.9
2,faixa_assinatura,11.76,9,0.227,3.6
3,inicio_antes_do_log,5.60,3,0.131,1.5
4,relogio_conectado,5.16,6,0.525,1.9
5,renovacao_desligada_no_envio,0.20,3,0.972,0.4


mediana de dias de assinatura no envio: {'controle': 278.0, 'conteudo': 267.0, 'desafio': 274.0, 'cupom': 281.5}


### Controle, engajamento e oferta

In [12]:
controle = c["grupo"] == "controle"
engaj_controle = int((controle & (c["abriu_push_b"] | c["abriu_email_b"] | c["clicou_b"])).sum())
achado("Consistência", "controle com abertura ou clique", f"{engaj_controle}", engaj_controle, "Nenhum")

clicou_sem_abrir = int((c["clicou_b"] & ~(c["abriu_push_b"] | c["abriu_email_b"])).sum())
achado("Consistência", "clicou sem abrir push nem e-mail", f"{clicou_sem_abrir}", clicou_sem_abrir, "Nenhum")
engaj = c[~controle].groupby("grupo")[["abriu_push_b", "abriu_email_b", "clicou_b"]].mean().mul(100).round(1)
display(engaj)

plano_log = eventos.dropna(subset=["plano"]).query("plano != ''").groupby("assinatura_id")["plano"].agg(["nunique", "first"])
assert (plano_log["nunique"] == 1).all()
c["plano_no_envio"] = c["assinatura_id"].map(plano_log["first"]).fillna(c["plano_atual"])
OFERTA = {"mensal": "30pct_3_meses", "anual": "1_mes_gratis"}
cupom = c["grupo"] == "cupom"
oferta_errada = int((cupom & (c["oferta"] != c["plano_no_envio"].map(OFERTA))).sum())
display(pd.crosstab(c.loc[cupom, "oferta"], c.loc[cupom, "plano_no_envio"]))
achado("Consistência", "oferta diferente da prevista para o plano na data do envio",
       f"{oferta_errada}; cada assinatura tem um único plano em todo o log", oferta_errada, "Nenhum")

,abriu_push_b,abriu_email_b,clicou_b
grupo,,,
conteudo,32.0,18.0,14.9
cupom,31.6,19.0,16.4
desafio,30.6,18.4,14.3


plano_no_envio,anual,mensal
oferta,,
1_mes_gratis,2052,0
30pct_3_meses,0,1420


### Quem entrou no sorteio sem pagar na hora do envio

O sorteio usou o dia inteiro (seção 1). Uma parte dos sorteados assinou depois das 10h ou já tinha saído antes das 10h:
nos grupos tratados, a "mensagem" chegou a quem ainda não era (ou já não era) assinante. Pelo sorteio, a proporção deve
ser a mesma nos quatro grupos.

In [13]:
ini = eventos[eventos["evento"] == "assinatura_iniciada"].drop_duplicates("user_id").set_index("user_id")["data_br"]
fim = eventos[eventos["evento"] == "assinatura_expirada"].drop_duplicates("user_id").set_index("user_id")["data_br"]
c["assinou_depois_do_envio"] = c["user_id"].map(ini).ge(ENVIO) & (c["inicio"] == DIA)
c["saiu_antes_do_envio"] = c["user_id"].map(fim).lt(ENVIO) & (c["data_fim"] == DIA)
fora_janela = c.groupby("grupo")[["assinou_depois_do_envio", "saiu_antes_do_envio"]].sum().loc[GRUPOS]
display(fora_janela)
n_fora = int((c["assinou_depois_do_envio"] | c["saiu_antes_do_envio"]).sum())
n_fora_trat = int(((c["assinou_depois_do_envio"] | c["saiu_antes_do_envio"]) & ~controle).sum())
achado("Consistência", "sorteado que assinou depois do envio ou saiu antes dele",
       f"{n_fora} contas ({n_fora_trat} nos grupos tratados): {int(c['assinou_depois_do_envio'].sum())} assinaram depois das 10h "
       f"de 10/08 e {int(c['saiu_antes_do_envio'].sum())} expiraram antes das 10h; a mensagem não podia agir sobre elas",
       n_fora, "Analisar com todos (intenção de tratar) e, como sensibilidade, sem essas contas, aplicando a mesma regra "
       "nos quatro grupos (inclusive no controle)")

,assinou_depois_do_envio,saiu_antes_do_envio
grupo,,
controle,7,3
conteudo,13,0
desafio,8,1
cupom,13,3


## 4. Unicidade

In [14]:
dup_id = int(campanha["user_id"].duplicated().sum())
dup_linha = int(campanha.duplicated().sum())
achado("Unicidade", "user_id repetido ou linha inteira repetida", f"{dup_id} e {dup_linha}", dup_id + dup_linha, "Nenhum")

## 5. Temporalidade

Envio único às 10h de 10/08/2026 e corte em 30/09/2026: 51 dias de observação. No mensal cabe ao menos uma renovação;
no anual, só quem faz aniversário nesse intervalo pode sair por não renovar.

In [15]:
envios = c.loc[~controle, "enviado"].value_counts()
print("envios:", {str(k): v for k, v in envios.items()})
achado("Temporalidade", "envio fora de 10/08/2026 10:00", f"{int((~controle & (c['enviado'] != ENVIO)).sum())}",
       int((~controle & (c["enviado"] != ENVIO)).sum()), "Nenhum")

dias_obs = (CORTE - DIA).days + 1
anual_total = int((c["plano_atual"] == "anual").sum())
anual_janela = len(na_janela)
achado("Temporalidade", "janela entre o envio e o corte",
       f"{dias_obs} dias (10/08 a 30/09); dos {anual_total:,} anuais, só {anual_janela:,} fazem aniversário nesse intervalo e "
       "podem sair por não renovar; a janela é menor que os 60 dias do modelo de saída",
       anual_total - anual_janela,
       "Medir o efeito em até 51 dias e separar mensal de anual; no anual, usar só quem tem aniversário na janela "
       "(ou a renovação desligada como desfecho intermediário)")

achado("Temporalidade", "abertura e clique sem data",
       "abriu_push, abriu_email e clicou não dizem quando aconteceram: podem ser posteriores ao corte", N,
       "Não usar abertura e clique como preditor de saída nem como filtro do efeito (são consequência do tratamento); "
       "usar só para descrever a atenção à mensagem")

envios: {'2026-08-10 10:00:00': 10541}


## 6. Resumo

In [16]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / "qualidade_campanha_ago26.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 14, 'atenção': 4, 'esperado': 2}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,"vazio disfarçado (nan, null, -)",ok,0,0,Nenhum,0.00
1,Completude,"envio, canal ou título vazio nos grupos tratad...",ok,0,0,Nenhum,0.00
2,Completude,"controle sem envio, canal, título nem oferta",esperado,"as 3,579 linhas do controle, nas quatro coluna...",0,Nenhum: é o desenho do experimento,0.00
3,Completude,"contas de teste, cópias ou sem cadastro no sor...",ok,teste: 0; cópias: 0; sem cadastro: 0,0,Se houver: excluir,0.00
4,Completude,população do sorteio contra os pagantes de 10/...,esperado,"as 14.120 contas estão entre os 14,122 que pag...",0,Nenhum: 2 contas em 14.122 não mudam o resulta...,0.00
5,Completude,cupom prometido pelo enunciado não aparece nas...,atenção,"0 cobranças de R$ 27,93 depois do envio (1,420...",3472,Medir o efeito do cupom como efeito da mensage...,24.59
6,Validade,"grupo, canal, oferta e engajamento fora do dom...",ok,0,0,Nenhum,0.00
7,Validade,título diferente do enunciado para o grupo,ok,0,0,Nenhum,0.00
8,Validade,enviado_em que não converte,ok,"0; padrões: {'9999-99-99 99:99:99': 10541, '':...",0,Nenhum,0.00
9,Validade,user_id fora do padrão,ok,0,0,Nenhum,0.00


In [17]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
